# Pilot: does the model separate *seeing* from *doing*?
**Jackfruit — Days 1–3 (Pranav K).** Runtime → Change runtime type → **T4 GPU** before running.

Goal: for each story, compare the model's beliefs after a **see** version vs a **do** version.
- **see** ("the road was wet"): belief in the cause (rain) *and* the effect (skid) should rise
- **do** ("someone hosed the road"): belief in the effect should rise, belief in the cause should **not**

If the model shows this gap, it has a see/do reference we can compare activation edits against later.

In [ ]:
!pip install -q "transformer_lens<4" pandas  # v4 removed HookedTransformer; pin to 3.x

In [ ]:
import gc, torch, pandas as pd
from transformer_lens import HookedTransformer
torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## 1. Gemma access (only needed for Gemma)
Gemma is gated: accept the licence at huggingface.co/google/gemma-2-2b, then log in with a **read** token.
Skip this cell if you only test Pythia and Qwen.

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token

## 2. Belief-scoring function
Belief = log P(" Yes") − log P(" No") at the answer position. Positive = model leans Yes.

A short few-shot prefix teaches the base model the Q/A format, so it puts its probability on Yes/No.

In [ ]:
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)

def make_prompt(story, question):
    return f"{FEWSHOT}Story: {story}\nQuestion: {question}\nAnswer:"

def get_yes_no_ids(model):
    ids = {}
    for w in [" Yes", " No"]:
        toks = model.to_tokens(w, prepend_bos=False)[0]
        assert len(toks) == 1, f"{w!r} is {len(toks)} tokens for this model"
        ids[w] = toks.item()
    return ids[" Yes"], ids[" No"]

def belief(model, story, question):
    """log P(Yes) - log P(No) for the next token."""
    yes_id, no_id = get_yes_no_ids(model)
    tokens = model.to_tokens(make_prompt(story, question))
    logits = model(tokens)[0, -1]
    logp = torch.log_softmax(logits.float(), dim=-1)
    return (logp[yes_id] - logp[no_id]).item()

## 3. Pilot stories
Each item has a neutral **base** context (B not mentioned), a **see** version and a **do** version,
plus a question about the upstream cause and the downstream effect.

👉 **Agree this format with Pranav P now** so his stories drop straight in (a CSV with these columns works: `pd.read_csv(...).to_dict("records")`).

In [ ]:
STORIES = [
    dict(id="rain_road",
         base="It was a cloudy afternoon. A car drove down the hill on Main Street.",
         see ="It was a cloudy afternoon. A car drove down the hill on Main Street. The road was wet.",
         do  ="It was a cloudy afternoon. A street cleaner sprayed Main Street with water, so the road was wet. A car drove down the hill on Main Street.",
         q_up="Did it rain?", q_down="Did the car skid?"),
    dict(id="fire_smoke",
         base="It was a quiet evening at the campsite. A ranger was walking nearby.",
         see ="It was a quiet evening at the campsite. A ranger was walking nearby. There was smoke rising from the trees.",
         do  ="It was a quiet evening at the campsite. A film crew released smoke from a machine, so there was smoke rising from the trees. A ranger was walking nearby.",
         q_up="Was there a fire?", q_down="Did the ranger see smoke?"),
    dict(id="flu_fever",
         base="Ravi went to work on Monday. In the afternoon his manager checked on him.",
         see ="Ravi went to work on Monday. He had a high fever. In the afternoon his manager checked on him.",
         do  ="Ravi went to work on Monday. He sat in a sauna for an hour, so his temperature was very high. In the afternoon his manager checked on him.",
         q_up="Did Ravi have an infection?", q_down="Did Ravi feel hot?"),
    dict(id="blicket",  # made-up names: no world knowledge to lean on
         base="In the lab there is a blicket and a dax. When the blicket glows, the dax hums. When the dax hums, the lamp flickers.",
         see ="In the lab there is a blicket and a dax. When the blicket glows, the dax hums. When the dax hums, the lamp flickers. Today the dax was humming.",
         do  ="In the lab there is a blicket and a dax. When the blicket glows, the dax hums. When the dax hums, the lamp flickers. Today a technician switched the dax on by hand, so it was humming.",
         q_up="Was the blicket glowing?", q_down="Did the lamp flicker?"),
]

## 4. Sanity check
Before trusting any numbers: obvious questions should get clearly positive / negative beliefs.
If these fail, fix the prompt format before going further.

In [ ]:
SANITY = [
    ("The door was locked and nobody had the key.", "Could anyone open the door?", "No"),
    ("Priya ate a huge lunch and felt full.",        "Was Priya full?",              "Yes"),
    ("The glass fell onto the stone floor and shattered.", "Did the glass break?",  "Yes"),
    ("The shop was closed all day on Sunday.",        "Was the shop open on Sunday?", "No"),
]
def sanity_check(model):
    ok = 0
    for story, q, want in SANITY:
        b = belief(model, story, q)
        got = "Yes" if b > 0 else "No"
        ok += got == want
        print(f"{b:+6.2f}  want {want:3}  {q}")
    print(f"{ok}/{len(SANITY)} correct")

## 5. Run the see/do test
For each story: Δ = belief(condition) − belief(base).

| | Δ upstream (cause) | Δ downstream (effect) |
|---|---|---|
| **see** should be | up | up |
| **do** should be | ≈ 0 | up |

**gap = Δup_see − Δup_do** — the key number. Clearly positive = the model separates see from do.

In [ ]:
def run_pilot(model, model_name):
    rows = []
    for s in STORIES:
        b = {q: belief(model, s["base"], s[q]) for q in ["q_up", "q_down"]}
        r = dict(model=model_name, id=s["id"])
        for cond in ["see", "do"]:
            r[f"up_{cond}"]   = belief(model, s[cond], s["q_up"])   - b["q_up"]
            r[f"down_{cond}"] = belief(model, s[cond], s["q_down"]) - b["q_down"]
        r["gap"] = r["up_see"] - r["up_do"]
        rows.append(r)
    return pd.DataFrame(rows)

def load(name):
    return HookedTransformer.from_pretrained_no_processing(name, device=DEVICE, dtype=torch.float32)

def free(model):
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
MODELS = ["pythia-1.4b", "Qwen/Qwen2.5-1.5B", "google/gemma-2-2b"]  # drop gemma if no HF access

results = []
for name in MODELS:
    print(f"\n===== {name} =====")
    model = load(name)
    sanity_check(model)
    df = run_pilot(model, name)
    display(df.round(2))
    results.append(df)
    free(model)

all_df = pd.concat(results)
all_df.to_csv("pilot_results.csv", index=False)

## 6. Summary per model

In [ ]:
summary = all_df.groupby("model")[["up_see", "up_do", "down_see", "down_do", "gap"]].mean().round(2)
summary["stories_with_gap>1"] = all_df.groupby("model")["gap"].apply(lambda g: (g > 1).sum())
summary

## How to read this (Day 4 decision)
- **Good model:** sanity check passes; `up_see` clearly > 0; `up_do` near 0; `down_*` both > 0; gap > ~1 on most stories.
- Pick the **smallest** model that does this — it's the model for the rest of the project.
- **Watch out:** if `up_do` is strongly **negative**, the model may be "explaining away" (the do-story names a different cause, e.g. the hose), not ignoring the cause. Note it — it matters for how Pranav P writes the do versions.
- If no model shows a gap → switch to the fallback framing on Day 4.